## Planning:
- Load data
- Create sample FE - sum of boolean and proportion of boolean and Digit Position
- Create TE in run.cv
- Implement for XGBoost

## Loading Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"One Dollar"

In [5]:
train = pd.read_csv(competition_path/"data/train.csv")
test = pd.read_csv(competition_path/"data/test.csv")
submit = pd.read_csv(competition_path/"data/sample_submission.csv")

## Data Transformation

In [28]:
# ln_var = ['allelectrons_Total','density_Total','atomicweight_Average']

In [29]:
'''
for col in ln_var:
    train[f'ln_{col}'] = np.log1p(train[col])
    test[f'ln_{col}'] = np.log1p(test[col])
'''

"\nfor col in ln_var:\n    train[f'ln_{col}'] = np.log1p(train[col])\n    test[f'ln_{col}'] = np.log1p(test[col])\n"

In [30]:
# train_cl = train.drop(columns= ln_var). copy()
# test_cl = test.drop(columns=ln_var).copy()

In [8]:
train_cl = train[train["allelectrons_Total"] <= test["allelectrons_Total"].max()].reset_index(drop=True).copy()
train_cl = train_cl[train_cl["atomicweight_Average"] <= test["atomicweight_Average"].max()].reset_index(drop=True)
test_cl = test.copy()

## Linear Regression

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import median_absolute_error

In [11]:
# 1. Define features and target (update 'Hardness' if your target column name differs)
target_col = "Hardness"
features = [col for col in train_cl.columns if col not in ["id", target_col]]

X = train_cl[features]
y = train_cl[target_col]
X_test = test_cl[features]

In [12]:
# 2. Build a pipeline (scaling helps linear models converge and stabilize weights)
model = make_pipeline(StandardScaler(), LinearRegression())

In [13]:
# 3. Set up 5-Fold Cross-Validation
kf = KFold(n_splits=3, shuffle=True, random_state=42)

In [14]:
# Note: scikit-learn negates error metrics in cross_val_score so higher = better
scores = cross_val_score(
    model, 
    X, 
    y, 
    cv=kf, 
    scoring="neg_median_absolute_error"
)

medae_scores = -scores
print(f"Fold MedAE scores: {np.round(medae_scores, 4)}")
print(f"Mean CV MedAE:     {medae_scores.mean():.4f} ± {medae_scores.std():.4f}")

Fold MedAE scores: [1.1131 1.1132 1.1172]
Mean CV MedAE:     1.1145 ± 0.0019


In [15]:
# 4. Fit on full training data and generate test predictions
model.fit(X, y)
test_preds = model.predict(X_test)

In [16]:
submit.head()

,id,Hardness
0,15000,4.421
1,15001,4.421
2,15002,4.421
3,15003,4.421
4,15004,4.421


In [17]:
submission = submit.copy()

In [18]:
submission[target_col] = test_preds

In [19]:
submission.to_csv(competition_path/"submission_2_lin.csv",index=False)

In [20]:
submission.head()

,id,Hardness
0,15000,4.077204
1,15001,4.739746
2,15002,5.018953
3,15003,3.575864
4,15004,5.099722


In [21]:
scores = cross_val_score(
    model, 
    X, 
    y, 
    cv=kf, 
    scoring="neg_root_mean_squared_error"
)

rmse_scores = -scores
print(f"Fold RMSE scores: {np.round(rmse_scores, 4)}")
print(f"Mean CV RMSE:     {rmse_scores.mean():.4f} ± {rmse_scores.std():.4f}")

Fold RMSE scores: [1.5128 1.5128 1.5109]
Mean CV RMSE:     1.5122 ± 0.0009


## XGBoost

In [22]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

# 1. Initialize XGBRegressor with baseline tabular parameters
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# 2. Cross-validate using your existing kf, X, and y
scores = cross_val_score(
    xgb_model,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse_scores = -scores
print(f"Fold RMSE scores: {np.round(rmse_scores, 4)}")
print(f"Mean CV RMSE:     {rmse_scores.mean():.4f} ± {rmse_scores.std():.4f}")

Fold RMSE scores: [1.2681 1.2577 1.2484]
Mean CV RMSE:     1.2581 ± 0.0080


In [23]:
# 3. Fit on full training data and generate submission
xgb_model.fit(X, y)
test_preds_xgb = xgb_model.predict(X_test)

submission_xgb = submit.copy()
submission_xgb[target_col] = test_preds_xgb
submission_xgb.to_csv(competition_path / "submission_5_xgb.csv", index=False)

### Adding Feature

In [24]:
def engineer_mohs_features(train_df: pd.DataFrame, test_df: pd.DataFrame, target_col: str = "Hardness"):
    train = train_df.copy()
    test = test_df.copy()
    
    # Identify original numerical feature columns (excluding id and target)
    base_num_cols = [
        col for col in train.columns 
        if col not in ["id", target_col] and pd.api.types.is_numeric_dtype(train[col])
    ]
    
    eps = 1e-5  # Small constant to prevent division by zero

    for df in (train, test):
        # 1. Physical & Chemical Ratio / Difference Features
        # Electron density & mass-to-electron relationships
        df["electron_density_ratio"] = df["allelectrons_Total"] / (df["density_Total"] + eps)
        df["mass_per_electron"] = df["atomicweight_Average"] / (df["allelectrons_Total"] + eps)
        df["allelectrons_per_mass"] = df["allelectrons_Total"] / (df["atomicweight_Average"] + eps)
        
        # Atomic radius & bond packing ratios (covalent vs. van der Waals radius)
        df["cov_to_vdw_ratio"] = df["R_cov_element_Average"] / (df["R_vdw_element_Average"] + eps)
        df["vdw_cov_diff"] = df["R_vdw_element_Average"] - df["R_cov_element_Average"]
        df["cov_volume_proxy"] = df["R_cov_element_Average"] ** 3
        df["vdw_volume_proxy"] = df["R_vdw_element_Average"] ** 3
        df["packing_fraction_proxy"] = df["cov_volume_proxy"] / (df["vdw_volume_proxy"] + eps)
        
        # Density discrepancies (Total vs. Average)
        df["density_diff"] = df["density_Total"] - df["density_Average"]
        df["density_ratio"] = df["density_Total"] / (df["density_Average"] + eps)
        df["estimated_atom_count"] = df["density_Total"] / (df["density_Average"] + eps)
        
        # Electronegativity & ionization energy interactions (bond strength proxies)
        df["ion_to_elneg_ratio"] = df["ionenergy_Average"] / (df["el_neg_chi_Average"] + eps)
        df["bond_polarity_proxy"] = df["el_neg_chi_Average"] * df["ionenergy_Average"]
        df["charge_radius_ratio"] = df["zaratio_Average"] / (df["R_cov_element_Average"] + eps)

        # 2. Row-Wise Zero-Count & Missing/Extreme Indicators
        df["zero_count"] = (df[base_num_cols] == 0).sum(axis=1)
        df["has_any_zero"] = (df["zero_count"] > 0).astype(int)

        # 3. Discretization / Decimal Digit Extraction
        # Synthetic tabular datasets often preserve rounding/digit artifacts from lookup tables
        for col in ["density_Total", "atomicweight_Average", "ionenergy_Average"]:
            df[f"{col}_dec1"] = np.floor((df[col] * 10) % 10).astype(int)
            df[f"{col}_dec2"] = np.floor((df[col] * 100) % 10).astype(int)

    # 4. Frequency Encoding (computed on combined train + test features only — zero target leakage)
    full_features = pd.concat([train[base_num_cols], test[base_num_cols]], axis=0, ignore_index=True)
    
    for col in base_num_cols:
        freq_map = full_features[col].value_counts()
        train[f"{col}_freq"] = train[col].map(freq_map).fillna(0)
        test[f"{col}_freq"] = test[col].map(freq_map).fillna(0)

    return train, test

In [25]:
# --- Usage ---
train_fe, test_fe = engineer_mohs_features(train, test, target_col="Hardness")

features = [col for col in train_fe.columns if col not in ["id", "Hardness"]]
X = train_fe[features]
y = train_fe["Hardness"]
X_test = test_fe[features]

print(f"Original features: {len(train.columns) - 2} -> Engineered features: {len(features)}")

Original features: 11 -> Engineered features: 44


## Training 5 FOLD RMSE

In [26]:
from sklearn.metrics import root_mean_squared_error, median_absolute_error
from xgboost import XGBRegressor

# 1. Set up 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

fold_rmse = []
fold_medae = []

# 2. Run Out-of-Fold (OOF) Training & Validation
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), start=1):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    model = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=10,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=1.0,
        reg_lambda=5.0,
        objective="reg:squarederror",  # Switch to "reg:absoluteerror" if Kaggle LB uses MedAE
        early_stopping_rounds=100,
        random_state=42 + fold,
        n_jobs=-1
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )
    
    val_preds = model.predict(X_va)
    oof_preds[val_idx] = val_preds
    test_preds += model.predict(X_test) / kf.n_splits
    
    rmse = root_mean_squared_error(y_va, val_preds)
    medae = median_absolute_error(y_va, val_preds)
    fold_rmse.append(rmse)
    fold_medae.append(medae)
    
    print(f"Fold {fold} | RMSE: {rmse:.4f} | MedAE: {medae:.4f} | Best Iter: {model.best_iteration}")



Fold 1 | RMSE: 1.2618 | MedAE: 0.7189 | Best Iter: 915
Fold 2 | RMSE: 1.2563 | MedAE: 0.7055 | Best Iter: 1327
Fold 3 | RMSE: 1.2335 | MedAE: 0.7087 | Best Iter: 1031
Fold 4 | RMSE: 1.2277 | MedAE: 0.7260 | Best Iter: 1463
Fold 5 | RMSE: 1.2704 | MedAE: 0.7582 | Best Iter: 1145


In [27]:
# 3. Overall OOF Metrics
print("-" * 50)
print(f"Overall OOF RMSE:  {root_mean_squared_error(y, oof_preds):.4f} (Mean: {np.mean(fold_rmse):.4f})")
print(f"Overall OOF MedAE: {median_absolute_error(y, oof_preds):.4f} (Mean: {np.mean(fold_medae):.4f})")



--------------------------------------------------
Overall OOF RMSE:  1.2500 (Mean: 1.2499)
Overall OOF MedAE: 0.7236 (Mean: 0.7235)


In [28]:
# 4. Post-process (clip to valid Mohs Hardness [1.0, 10.0]) and save submission
submission_xgb_fe = submit.copy()
submission_xgb_fe[target_col] = np.clip(test_preds, 1.0, 10.0)
submission_xgb_fe.to_csv(competition_path / "submission_6_xgb_fe.csv", index=False)
submission_xgb_fe.head()

,id,Hardness
0,15000,5.596671
1,15001,4.527040
2,15002,5.643884
3,15003,3.840838
4,15004,6.013750


## Training 5 fold AbsMed

In [29]:
# 1. Set up 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

fold_rmse = []
fold_medae = []

# 2. Run Out-of-Fold (OOF) Training & Validation
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), start=1):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    model = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=10,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=1.0,
        reg_lambda=5.0,
        objective="reg:absoluteerror",  # Switch to "reg:absoluteerror" if Kaggle LB uses MedAE
        early_stopping_rounds=100,
        random_state=42 + fold,
        n_jobs=-1
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )
    
    val_preds = model.predict(X_va)
    oof_preds[val_idx] = val_preds
    test_preds += model.predict(X_test) / kf.n_splits
    
    rmse = root_mean_squared_error(y_va, val_preds)
    medae = median_absolute_error(y_va, val_preds)
    fold_rmse.append(rmse)
    fold_medae.append(medae)
    
    print(f"Fold {fold} | RMSE: {rmse:.4f} | MedAE: {medae:.4f} | Best Iter: {model.best_iteration}")



Fold 1 | RMSE: 1.3024 | MedAE: 0.5942 | Best Iter: 1997
Fold 2 | RMSE: 1.2952 | MedAE: 0.6004 | Best Iter: 1962
Fold 3 | RMSE: 1.2676 | MedAE: 0.5694 | Best Iter: 1997
Fold 4 | RMSE: 1.2667 | MedAE: 0.5730 | Best Iter: 1999
Fold 5 | RMSE: 1.2997 | MedAE: 0.6182 | Best Iter: 1944


In [30]:
# 3. Overall OOF Metrics
print("-" * 50)
print(f"Overall OOF RMSE:  {root_mean_squared_error(y, oof_preds):.4f} (Mean: {np.mean(fold_rmse):.4f})")
print(f"Overall OOF MedAE: {median_absolute_error(y, oof_preds):.4f} (Mean: {np.mean(fold_medae):.4f})")


--------------------------------------------------
Overall OOF RMSE:  1.2864 (Mean: 1.2863)
Overall OOF MedAE: 0.5895 (Mean: 0.5910)


In [31]:
# 4. Post-process (clip to valid Mohs Hardness [1.0, 10.0]) and save submission
submission_xgb_fe = submit.copy()
submission_xgb_fe[target_col] = np.clip(test_preds, 1.0, 10.0)
submission_xgb_fe.to_csv(competition_path / "submission_7_xgb_fe.csv", index=False)
submission_xgb_fe.head()

,id,Hardness
0,15000,6.081442
1,15001,4.802617
2,15002,5.982893
3,15003,4.302235
4,15004,6.064949
